In [ ]:
import os
import pickle
import re
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

from connectDriveCloud import authenticate_google_drive, load_pickle_content, get_files
from distances import fidelityCalc, traceDist, getHellinger, compareChisquare, jensenShannonDivergence

# Load CSV results

In [ ]:
mutants_file_path = "results_custom_brisbane/results_normal.csv"
equiv_file_path =  "results_custom_brisbane/results_equiv.csv"

column_names = ['Name', 'Input', 'Ideal_chisquare', 'Ideal_hellinger', 'Ideal_jensenshannon', 'Ideal_trace', 'Ideal_fidelity',
                'Ideal_expectation', 'Killed_IC', 'Killed_IH', 'Killed_IJ', 'Killed_IT', 'Killed_IF', 'Killed_IE']
    
mutants_df = pd.read_csv(mutants_file_path, names=column_names, header=0)
equiv_df = pd.read_csv(equiv_file_path, names=column_names, header=0)

# Display the DataFrame to verify
print(mutants_df.head())
print(equiv_df.head())

# Print number of mutants

In [ ]:
unique_count = mutants_df['Name'].nunique()
print("Number of executions in the mutant set: ", len(mutants_df))
print("Number of mutants in the mutant set: ", unique_count)


unique_count = equiv_df['Name'].nunique()
print("Number of executions in the equivalent set: ", len(equiv_df))
print("Number of mutants in the equivalent set: ", unique_count)


# Check for Equivalence 

In [ ]:
trace_error_rounded = 1E-13
fidelity_error_rounded = 1E-14

In [ ]:
def classify_mutants(mutants_df):
    
    # Find killed mutants based on Ideal_expectation
    killed = mutants_df[mutants_df['Ideal_expectation'] != 0]
    killed = killed['Name'].unique()

    # Find mixed mutants based on Ideal_expectation and Ideal_fidelity
    df_mutant_candidates = mutants_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).all())
    df_filtered = df_mutant_candidates[(1 - df_mutant_candidates['Ideal_fidelity'] > fidelity_error_rounded)]
    mixed = df_filtered['Name'].unique()

    # Find survived mutants that are neither killed nor mixed
    survived = mutants_df[~mutants_df['Name'].isin(killed) & ~mutants_df['Name'].isin(mixed)]
    survived = survived['Name'].unique()
    
    return killed, survived, mixed


In [ ]:
def check_sizes(mutants_df, killed, survived, mixed):
    print("Total mutants: ", len(mutants_df['Name'].unique()))
    print("Killed mutants: ", len(killed))
    print("Equivalent mutants: ", len(survived))
    print("Mixed mutants: ", len(mixed))
    print("Check total: ", len(mixed) + len(killed) + len(survived))

In [ ]:
killed, survived, mixed = classify_mutants(equiv_df)
print("Mutants in equivalent set:")
total = len(equiv_df['Name'].unique())
print("Total mutants", total)
#print(killed)
#print(survived)
#print(mixed)
#check_sizes(equiv_df, killed, survived, mixed)
killed_or_mixed = set(killed).union(set(mixed))
#print(killed_or_mixed)
print("Killed mutants", len(killed_or_mixed))
percentage = round(len(killed_or_mixed) / total * 100, 2)
print("Percentage to move: " + str(percentage) + "%")

print("")

killed, survived, mixed = classify_mutants(mutants_df)
print("Mutants in normal set:")
total = len(mutants_df['Name'].unique())
print("Total mutants", total)
#print("Killed by expectation", killed)
#print(survived)
print("Survived mutants", len(survived))
#print(mixed)

killed_or_mixed = set(killed).union(set(mixed))

percentage = round(len(survived) / total * 100, 2)
print("Percentage to move: " + str(percentage) + "%")

#print(killed_or_mixed)

#check_sizes(mutants_df, killed, survived, mixed)

In [ ]:
def further_analysis(mutants_df):
    # Shows that expectation > fidelity and trace
    df_filtered = mutants_df[(mutants_df['Ideal_expectation'] != 0)]
    nb_non_equivalent_executions = len(df_filtered)
    
    fidelity_mutants = df_filtered[(1 - df_filtered['Ideal_fidelity'] > fidelity_error_rounded)]
    fidelity_equivalence = len(fidelity_mutants) == nb_non_equivalent_executions
    
    trace_mutants = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
    trace_equivalence = len(trace_mutants) == nb_non_equivalent_executions
    
    # Find rows in expectation_mutants that are not in trace_mutants
    # only_in_expectation_mutants = (df_filtered)[~df_filtered.index.isin(trace_mutants.index)]
    # print(only_in_expectation_mutants)
    
    print("Does fidelity detect the mutation when expectation does?", fidelity_equivalence)
    print("Does trace distance detect the mutation when expectation does?", trace_equivalence)
    
    df_filtered_names = set(df_filtered['Name'].unique())
    trace_mutants_names = set(trace_mutants['Name'].unique())
    only_in_df_filtered = df_filtered_names - trace_mutants_names
    only_in_trace_mutants = trace_mutants_names - df_filtered_names
    both_sets_empty = not only_in_df_filtered and not only_in_trace_mutants
    print("Are all individual mutants detected by both trace distance and expectation?", both_sets_empty)
    
    # Shows that fidelity > trace and  expectation != fidelity
    df_filtered = mutants_df[(mutants_df['Ideal_expectation'] == 0) & (1 - mutants_df['Ideal_fidelity'] > fidelity_error_rounded)]
    df_fidelity_ideal = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
    df_ideal = mutants_df[(mutants_df['Ideal_expectation'] == 0) & (mutants_df['Ideal_trace'] > trace_error_rounded)]
    
    trace_equivalence = len(df_fidelity_ideal) == len(df_ideal)
    
    print("Are mutants discover by trace distance also detected by fidelity?", trace_equivalence)

In [ ]:
print("Equivalent set analysis:")
further_analysis(equiv_df)
print("\nNormal set analysis:")
further_analysis(mutants_df)

# Define thresholds for output distribution metrics

In [ ]:
def get_theoretical_distribution(density_matrix, nb_shots):
    # Extract the diagonal elements (probabilities)
    probabilities = np.real(np.diag(density_matrix.data))
    probabilities = probabilities.copy()  # Create a writable copy

    # Normalize probabilities
    total_prob = np.sum(probabilities)
    if total_prob > 0:
        probabilities /= total_prob  # Normalize to ensure probabilities sum to 1

    # Calculate the number of qubits from the size of the density matrix
    num_qubits = int(np.log2(len(probabilities)))

    # Generate state labels for computational basis states
    state_labels = [format(i, f'0{num_qubits}b') for i in range(2 ** num_qubits)]

    # Calculate the initial counts for each state based on the number of shots
    initial_counts = np.array([prob * nb_shots for prob in probabilities])  # Use NumPy for easy operations

    # Round counts to nearest int and convert to Python int
    counts = [int(round(count)) for count in initial_counts]

    # Ensure the total counts sum to nb_shots
    total_counts = sum(counts)

    if total_counts != nb_shots:
        # Calculate the difference
        difference = nb_shots - total_counts

        # Calculate the adjustment needed
        adjustment_indices = np.argsort(initial_counts - counts)[:abs(difference)]

        # Adjust counts
        if difference > 0:
            for idx in adjustment_indices:
                counts[idx] += 1  # Increment counts for excess shots
        else:
            for idx in adjustment_indices:
                counts[idx] -= 1  # Decrement counts for excess counts

    # Create a dictionary mapping states to their corresponding counts
    count_dict = {state: int(count) for state, count in zip(state_labels, counts)}

    return count_dict

In [ ]:
def get_ideal_thresholds(oracle_data):
    column_names = ['Name', 'Input', 'Chisquare', 'Hellinger', 'Jensenshannon']
    # Convert the oracle_data list of dictionaries into a lookup dictionary for faster access
    oracle_lookup = {item['Input']: item for item in oracle_data}

    results = []
    for key, value in oracle_lookup.items():

        # Theoretical distribution
        theoretical_distribution = get_theoretical_distribution(value['Ideal_density_matrix'], 10000)

        # Observed distribution (from Ideal_output_distribution)
        observed_distribution = value['Ideal_output_distribution']

        hellinger = getHellinger(theoretical_distribution, observed_distribution)
        jensen = jensenShannonDivergence(theoretical_distribution, observed_distribution)
        chisquare = compareChisquare(theoretical_distribution, observed_distribution)

        name = value['Name'].split('/')[-1]
        new_line = {'Name': name, 'Input': value['Input'], 'Chisquare': chisquare, 'Hellinger': hellinger,
                    'Jensenshannon': jensen}
        results.append(new_line)

    # Convert the results list of dictionaries to a DataFrame
    results_df = pd.DataFrame(results, columns=column_names)
    return results_df

In [ ]:
def process_files(service, origin_id):
    origin_files = get_files(service, origin_id)
    df_total = pd.DataFrame(
        columns=['Name', 'Input', 'Chisquare', 'Hellinger', 'Jensenshannon'])
    for item in tqdm(origin_files, desc="Checking results..."):
        filename = item['name']
        file_id = item['id']
        if filename.endswith('.pkl'):
            try:
                oracle_pkl = load_pickle_content(service, file_id)
                if isinstance(oracle_pkl, list):
                    new_df = get_ideal_thresholds(oracle_pkl)
                    df_total = pd.concat([df_total, new_df], ignore_index=True)
                else:
                    print(f"Pickle file should contain a List instead of a {type(oracle_pkl)}.")
            except pickle.UnpicklingError:
                print(f'Error unpickling file: {filename}')
            except Exception as e:
                print(f'Error processing file {filename}: {str(e)}')
    return df_total

In [ ]:
origin_id = "1MTTleRgnFJ2UnYmbpzZoh2ndmWBJ3YJk"
service = authenticate_google_drive()
df_total = process_files(service, origin_id)

In [ ]:
mean_values = df_total.iloc[:, 2:].mean()
print('Mean: ')
print(mean_values)
std_dev = df_total.iloc[:, 2:].std()
print('Standard deviation: ')
print(std_dev)
n = len(df_total)  # Number of observations
std_error = std_dev / np.sqrt(n)
print('Standard error: ')
print(std_error)
print('Threshold: ')
print(f"Chisquare: {mean_values['Chisquare'] + std_error['Chisquare']}")
print(f"Hellinger: {mean_values['Hellinger'] + std_error['Hellinger']}")
print(f"Jensenshannon: {mean_values['Jensenshannon'] + std_error['Jensenshannon']}")

# Killed flags

In [ ]:
def calculate_killed_flags(ideal, tolerance_values_ideal):
    """
    Determines the killed flags based on ideal and noisy values and tolerance values.
    """
    killed_flags = {}
    killed_flags['Killed_IF'] = ideal['fidelity'] < tolerance_values_ideal['fidelity']
    killed_flags['Killed_IT'] = ideal['trace'] > tolerance_values_ideal['trace']
    killed_flags['Killed_IH'] = ideal['hellinger'] > tolerance_values_ideal['hellinger']
    killed_flags['Killed_IC'] = ideal['chisquare'] < tolerance_values_ideal['chisquare']
    killed_flags['Killed_IJ'] = ideal['jensenshannon'] > tolerance_values_ideal['jensenshannon']
    killed_flags['Killed_IE'] = ideal['expectation'] > tolerance_values_ideal['expectation']
    return pd.DataFrame(killed_flags)

In [ ]:
# Define tolerance values
tolerance_values_ideal = {
    'fidelity': 1E-14, 
    'trace': 1E-13, 
    'hellinger': 0.13455009062719828,
    'jensenshannon': 0.11716009455796059,
    'chisquare': 0.318714816155845, #.0000001,
    'expectation': 0 
}


In [ ]:
results_df = equiv_df

In [ ]:
# Determine killed flags
killed_flags_df = calculate_killed_flags(
    ideal={'fidelity': results_df['Ideal_fidelity'], 'trace': results_df['Ideal_trace'], 'hellinger': results_df['Ideal_hellinger'],
            'chisquare': results_df['Ideal_chisquare'], 'jensenshannon': results_df['Ideal_jensenshannon'], 'expectation': results_df['Ideal_expectation']},
    tolerance_values_ideal=tolerance_values_ideal
)

# Fill the existing NaN columns in results_df with calculated flags
for flag in killed_flags_df.columns:
    results_df[flag] = killed_flags_df[flag]